In [ ]:
# ==============================================================================
# Cell 1: Setup - Download FSC Dataset + Clone Repo
# ==============================================================================
import os, subprocess

# Clone / update repo
REPO = '/content/NC-KWS-FineTuning'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/DrJinHoChoi/NC-KWS-FineTuning.git', REPO], check=True)
os.chdir(REPO)

# Download Fluent Speech Commands (FSC) from Zenodo
FSC_DIR = '/content/fluent_speech_commands_dataset'
if not os.path.exists(FSC_DIR):
    print('Downloading Fluent Speech Commands (~1.5GB)...')
    subprocess.run(['wget', '-q', '--show-progress',
                    'https://zenodo.org/records/3509828/files/fluent_speech_commands_dataset.tar.gz',
                    '-O', '/content/fsc.tar.gz'], check=True)
    subprocess.run(['tar', 'xzf', '/content/fsc.tar.gz', '-C', '/content/'], check=True)
    os.remove('/content/fsc.tar.gz')
    print('Done!')
else:
    print(f'FSC already at {FSC_DIR}')

# Verify
import sys
sys.path.insert(0, REPO)
from nanomamba import create_nc_tcn_20k
print(f'NC-TCN-20K loaded: {sum(p.numel() for p in create_nc_tcn_20k().parameters()):,} params')

# Check FSC structure
import pandas as pd
for split in ['train', 'valid', 'test']:
    csv_path = os.path.join(FSC_DIR, 'data', f'{split}_data.csv')
    df = pd.read_csv(csv_path)
    print(f'{split}: {len(df)} utterances')
print('Setup complete!')


In [ ]:
# ==============================================================================
# Cell 2: NC-SLU - Few-Shot Spoken Language Understanding
# ==============================================================================
# First sub-100K param SLU model with few-shot intent addition
# Dataset: Fluent Speech Commands (FSC) - 31 intents
# Protocol: 25 base intents -> incrementally add 6 new intents (20-shot)
# ==============================================================================

import os, sys, time, copy, random
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

sys.path.insert(0, '/content/NC-KWS-FineTuning')
from nanomamba import create_nc_tcn_20k

SR = 16000
MAX_LEN = SR * 3  # 3 seconds max
FSC_DIR = '/content/fluent_speech_commands_dataset'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Load FSC data ----
def load_fsc_split(split):
    csv_path = os.path.join(FSC_DIR, 'data', f'{split}_data.csv')
    df = pd.read_csv(csv_path)
    data = []
    for _, row in df.iterrows():
        # Intent = (action, object, location) tuple
        intent = f"{row['action']}_{row['object']}_{row['location']}"
        wav_path = os.path.join(FSC_DIR, row['path'])
        data.append({'intent': intent, 'path': wav_path,
                     'action': row['action'], 'object': row['object'],
                     'location': row['location'], 'text': row['transcription']})
    return data

def load_audio(path):
    w, sr = torchaudio.load(path)
    if sr != SR:
        w = torchaudio.functional.resample(w, sr, SR)
    a = w[0].numpy()
    if len(a) > MAX_LEN: a = a[:MAX_LEN]
    elif len(a) < MAX_LEN: a = np.pad(a, (0, MAX_LEN - len(a)))
    return a.astype(np.float32)

print('Loading FSC data...')
train_data = load_fsc_split('train')
val_data = load_fsc_split('valid')
test_data = load_fsc_split('test')

# Get unique intents
all_intents = sorted(set(d['intent'] for d in train_data))
print(f'Total unique intents: {len(all_intents)}')
for i, intent in enumerate(all_intents):
    count = sum(1 for d in train_data if d['intent'] == intent)
    if i < 10 or i >= len(all_intents) - 3:
        print(f'  [{i:2d}] {intent}: {count} train samples')
    elif i == 10:
        print(f'  ...')

# ---- Intent split: 25 base + 6 new ----
random.seed(42)
intents_shuffled = all_intents.copy()
random.shuffle(intents_shuffled)
BASE_INTENTS = intents_shuffled[:25]
NEW_INTENTS = intents_shuffled[25:]
N_SHOT = 20  # few-shot for new intents

print(f'\nBase intents ({len(BASE_INTENTS)}): {BASE_INTENTS[:5]}...')
print(f'New intents ({len(NEW_INTENTS)}): {NEW_INTENTS}')

# ---- Load audio by intent ----
print('\nLoading audio...')
def build_dataset(data_list, intents, max_per_intent=None):
    """Load audio grouped by intent."""
    result = defaultdict(list)
    for d in data_list:
        if d['intent'] in intents:
            if max_per_intent and len(result[d['intent']]) >= max_per_intent:
                continue
            try:
                result[d['intent']].append(load_audio(d['path']))
            except: continue
    return dict(result)

t0 = time.time()
# Base intents: full training data
base_train = build_dataset(train_data, BASE_INTENTS)
base_val = build_dataset(val_data, BASE_INTENTS)
base_test = build_dataset(test_data, BASE_INTENTS)

# New intents: N_SHOT for training, rest for testing
new_all_train = build_dataset(train_data, NEW_INTENTS)
new_train = {k: v[:N_SHOT] for k, v in new_all_train.items()}
new_test = build_dataset(test_data, NEW_INTENTS)

total_base = sum(len(v) for v in base_train.values())
total_new = sum(len(v) for v in new_train.values())
total_test = sum(len(v) for v in base_test.values()) + sum(len(v) for v in new_test.values())
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'  Base train: {total_base} samples across {len(base_train)} intents')
print(f'  New train: {total_new} samples ({N_SHOT}-shot x {len(new_train)} intents)')
print(f'  Test: {total_test} samples')

# ---- Audio Augmentation ----
def aug_time_stretch(a, rate):
    n = int(len(a) / rate)
    indices = np.linspace(0, len(a)-1, n).astype(np.float32)
    idx_floor = np.floor(indices).astype(int)
    idx_ceil = np.minimum(idx_floor + 1, len(a)-1)
    frac = indices - idx_floor
    stretched = a[idx_floor] * (1 - frac) + a[idx_ceil] * frac
    if len(stretched) > MAX_LEN: stretched = stretched[:MAX_LEN]
    elif len(stretched) < MAX_LEN: stretched = np.pad(stretched, (0, MAX_LEN - len(stretched)))
    return stretched.astype(np.float32)

def augment(a, bg_pool=None):
    out = a.copy()
    if np.random.random() < 0.5:
        out = aug_time_stretch(out, np.random.uniform(0.9, 1.1))
    if np.random.random() < 0.5:
        out = np.roll(out, np.random.randint(-4800, 4800))
    if np.random.random() < 0.5:
        db = np.random.uniform(-4, 4)
        out = out * (10 ** (db / 20))
    if np.random.random() < 0.3:
        ml = int(len(out) * np.random.uniform(0.05, 0.1))
        s = np.random.randint(0, len(out) - ml)
        out[s:s+ml] = 0.0
    out = out + np.random.randn(len(out)).astype(np.float32) * 0.003
    return out.astype(np.float32)

# ---- LoRA ----
class LoRALinear(nn.Module):
    def __init__(self, original, rank=4, alpha=8):
        super().__init__()
        self.original = original
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(original.in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, original.out_features))
    def forward(self, x):
        return self.original(x) + (x @ self.lora_A @ self.lora_B) * self.scaling

def label_smooth_ce(logits, targets, smoothing=0.1):
    log_probs = F.log_softmax(logits, dim=-1)
    nll = -log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
    smooth = -log_probs.mean(dim=-1)
    return ((1 - smoothing) * nll + smoothing * smooth).mean()

# ---- Train base SLU model ----
def train_base_slu(intents, train_data, val_data, epochs=30):
    n_cls = len(intents)
    model = create_nc_tcn_20k(n_classes=n_cls).to(DEVICE)
    print(f'  Base SLU model: {n_cls} intents, {sum(p.numel() for p in model.parameters()):,} params')

    X, Y = [], []
    for i, intent in enumerate(intents):
        for a in train_data.get(intent, []):
            X.append(a); Y.append(i)
    print(f'  Training: {len(X)} samples')

    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(len(X))
        eloss, nb = 0, 0
        for i in range(0, len(X), 32):
            bi = perm[i:i+32]
            batch = [augment(X[j]) if np.random.random() < 0.5 else X[j] for j in bi]
            bx = torch.stack([torch.from_numpy(a).float() for a in batch]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            loss = F.cross_entropy(model(bx), by)
            opt.zero_grad(); loss.backward(); opt.step()
            eloss += loss.item(); nb += 1
        sched.step()

        if (ep+1) % 10 == 0 or ep == 0:
            model.eval()
            vc, vt = 0, 0
            with torch.no_grad():
                for i, intent in enumerate(intents):
                    for a in val_data.get(intent, []):
                        x = torch.from_numpy(a).float().unsqueeze(0).to(DEVICE)
                        pred = model(x).argmax(-1).item()
                        vt += 1; vc += (pred == i)
            print(f'    Epoch {ep+1}/{epochs} loss={eloss/nb:.4f} val_acc={vc/max(vt,1)*100:.1f}%')

    return model

# ---- Evaluate ----
def evaluate_slu(model, intents, test_data):
    model.eval()
    correct, total = 0, 0
    per_intent = {}
    with torch.no_grad():
        for i, intent in enumerate(intents):
            ic, it = 0, 0
            for a in test_data.get(intent, []):
                x = torch.from_numpy(a).float().unsqueeze(0).to(DEVICE)
                pred = model(x).argmax(-1).item()
                it += 1; ic += (pred == i)
            per_intent[intent] = ic / max(it, 1)
            correct += ic; total += it
    return correct / max(total, 1), per_intent

# ---- NC-OPAL for few-shot intent addition ----
def ncopal_slu(base_model, base_intents, new_intents, new_train, base_train_sub,
               epochs=40, lr=3e-3, lambda_kd=2.0, kd_temp=3.0):
    model = copy.deepcopy(base_model)
    n_old = len(base_intents)
    n_new = len(new_intents)
    n_total = n_old + n_new
    d = model.classifier.in_features

    # Prototype-imprinted head
    emb_hook = {}
    def hook_fn(m, inp, out): emb_hook['e'] = inp[0].detach()
    def get_emb(audio):
        h = model.classifier.register_forward_hook(hook_fn)
        with torch.no_grad(): model(torch.from_numpy(audio).float().unsqueeze(0).to(DEVICE))
        h.remove()
        return emb_hook['e'].squeeze(0)

    old_head = model.classifier
    new_head = nn.Linear(d, n_total).to(DEVICE)
    with torch.no_grad():
        new_head.weight[:n_old] = old_head.weight
        new_head.bias[:n_old] = old_head.bias
        scale = old_head.weight.norm(dim=1).mean().item()
        for i, intent in enumerate(new_intents):
            embs = [get_emb(a) for a in new_train.get(intent, [])]
            if embs:
                proto = F.normalize(torch.stack(embs).mean(0), dim=0)
                new_head.weight[n_old+i] = proto * scale
                new_head.bias[n_old+i] = old_head.bias.mean().item()
    model.classifier = new_head

    # LoRA
    loras = []
    for block in model.blocks:
        for attr in ['in_proj', 'out_proj']:
            if hasattr(block, attr):
                lo = LoRALinear(getattr(block, attr), rank=4, alpha=8).to(DEVICE)
                setattr(block, attr, lo); loras.append(lo)

    # Teacher
    teacher = copy.deepcopy(base_model).to(DEVICE)
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False

    # Build training data
    X, Y = [], []
    for i, intent in enumerate(new_intents):
        li = n_old + i
        for a in new_train.get(intent, []):
            X.append(a); Y.append(li)
            for _ in range(8):
                X.append(augment(a)); Y.append(li)
    for i, intent in enumerate(base_intents):
        for a in base_train_sub.get(intent, [])[:15]:
            X.append(a); Y.append(i)
            X.append(augment(a)); Y.append(i)

    n_new_s = sum(1 for y in Y if y >= n_old)
    n_old_s = sum(1 for y in Y if y < n_old)
    print(f'  Train: {len(X)} (new={n_new_s}, old={n_old_s})')

    # Stage 1: Head only (10 epochs)
    for p in model.parameters(): p.requires_grad = False
    for p in new_head.parameters(): p.requires_grad = True
    opt1 = torch.optim.AdamW(new_head.parameters(), lr=lr*3, weight_decay=0.01)
    model.train()
    for ep in range(10):
        perm = np.random.permutation(len(X))
        for i in range(0, len(X), 16):
            bi = perm[i:i+16]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            loss = label_smooth_ce(model(bx), by, 0.1)
            opt1.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(new_head.parameters(), 1.0)
            opt1.step()
    print(f'  Stage 1 done (head-only, 10 epochs)')

    # Stage 2: LoRA + Head + KD (30 epochs)
    for m in loras:
        for p in m.parameters(): p.requires_grad = True
    opt2 = torch.optim.AdamW([
        {'params': [p for m in loras for p in m.parameters()], 'lr': lr},
        {'params': new_head.parameters(), 'lr': lr * 0.5},
    ], weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=30)

    for ep in range(30):
        perm = np.random.permutation(len(X))
        ecl, ekd, nbt = 0, 0, 0
        for i in range(0, len(X), 16):
            bi = perm[i:i+16]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            logits = model(bx)
            cls_loss = label_smooth_ce(logits, by, 0.05)

            kd_loss = torch.tensor(0.0).to(DEVICE)
            old_mask = by < n_old
            if old_mask.any():
                with torch.no_grad(): t_logits = teacher(bx[old_mask])
                s_base = logits[old_mask][:, :n_old] / kd_temp
                t_base = t_logits / kd_temp
                kd_loss = F.kl_div(F.log_softmax(s_base, -1),
                                   F.softmax(t_base, -1),
                                   reduction='batchmean') * (kd_temp**2)

            kd_w = lambda_kd * min(1.0, ep / 5.0)
            total = cls_loss + kd_w * kd_loss
            opt2.zero_grad(); total.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt2.step()
            ecl += cls_loss.item(); ekd += kd_loss.item(); nbt += 1
        sched.step()
        if (ep+1) % 10 == 0:
            print(f'    Epoch {ep+1}/30 cls={ecl/nbt:.4f} kd={ekd/nbt:.4f}')

    return model, base_intents + new_intents

# ---- Finetune baseline ----
def finetune_slu(base_model, base_intents, new_intents, new_train, base_train_sub, epochs=40, lr=3e-3):
    model = copy.deepcopy(base_model)
    n_old = len(base_intents)
    n_total = n_old + len(new_intents)
    d = model.classifier.in_features

    old_head = model.classifier
    new_head = nn.Linear(d, n_total).to(DEVICE)
    with torch.no_grad():
        new_head.weight[:n_old] = old_head.weight
        new_head.bias[:n_old] = old_head.bias
        nn.init.xavier_uniform_(new_head.weight[n_old:])
        new_head.bias[n_old:].zero_()
    model.classifier = new_head

    X, Y = [], []
    for i, intent in enumerate(new_intents):
        li = n_old + i
        for a in new_train.get(intent, []):
            X.append(a); Y.append(li)
            for _ in range(8): X.append(augment(a)); Y.append(li)
    for i, intent in enumerate(base_intents):
        for a in base_train_sub.get(intent, [])[:15]:
            X.append(a); Y.append(i)
            X.append(augment(a)); Y.append(i)

    for p in model.parameters(): p.requires_grad = True
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    model.train()
    for ep in range(epochs):
        perm = np.random.permutation(len(X))
        for i in range(0, len(X), 16):
            bi = perm[i:i+16]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            loss = F.cross_entropy(model(bx), by)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
    return model, base_intents + new_intents

# ====================
# MAIN EXPERIMENT
# ====================
print('\n' + '='*70)
print('  NC-SLU: Few-Shot Spoken Language Understanding')
print('  First sub-100K param SLU with few-shot intent addition')
print(f'  Base: {len(BASE_INTENTS)} intents | New: {len(NEW_INTENTS)} intents ({N_SHOT}-shot)')
print(f'  Device: {DEVICE}')
print('='*70)

# ---- Phase 1: Train base SLU model ----
print('\n[Phase 1] Training base SLU model on 25 intents...')
base_model = train_base_slu(BASE_INTENTS, base_train, base_val, epochs=30)
base_acc, base_per = evaluate_slu(base_model, BASE_INTENTS, base_test)
print(f'\nBase model accuracy: {base_acc*100:.1f}% on {len(BASE_INTENTS)} intents')

# ---- Phase 2: Few-shot intent addition ----
print('\n[Phase 2] Adding {len(NEW_INTENTS)} new intents ({N_SHOT}-shot)...')
# Subsample base train for rehearsal
base_train_sub = {k: v[:15] for k, v in base_train.items()}

results = {}

# Method 1: Finetune
print('\n--- Method: Finetune (all params) ---')
ft_model, ft_intents = finetune_slu(base_model, BASE_INTENTS, NEW_INTENTS, new_train, base_train_sub)
all_test_data = {**base_test, **new_test}
ft_acc, ft_per = evaluate_slu(ft_model, ft_intents, all_test_data)
ft_base_acc = np.mean([ft_per[i] for i in BASE_INTENTS])
ft_new_acc = np.mean([ft_per[i] for i in NEW_INTENTS])
results['Finetune'] = {'overall': ft_acc, 'base': ft_base_acc, 'new': ft_new_acc, 'per': ft_per}
print(f'  Overall: {ft_acc*100:.1f}% | Base: {ft_base_acc*100:.1f}% | New: {ft_new_acc*100:.1f}%')

# Method 2: NC-OPAL
print('\n--- Method: NC-OPAL ---')
opal_model, opal_intents = ncopal_slu(base_model, BASE_INTENTS, NEW_INTENTS, new_train, base_train_sub)
opal_acc, opal_per = evaluate_slu(opal_model, opal_intents, all_test_data)
opal_base_acc = np.mean([opal_per[i] for i in BASE_INTENTS])
opal_new_acc = np.mean([opal_per[i] for i in NEW_INTENTS])
results['NC-OPAL'] = {'overall': opal_acc, 'base': opal_base_acc, 'new': opal_new_acc, 'per': opal_per}
print(f'  Overall: {opal_acc*100:.1f}% | Base: {opal_base_acc*100:.1f}% | New: {opal_new_acc*100:.1f}%')

# ---- Summary ----
print('\n' + '='*70)
print('  NC-SLU RESULTS (NC-TCN-20K, 21,689 params)')
print('='*70)
print(f'{"Method":<20} {"Overall":>10} {"Base (25)":>10} {"New (6)":>10} {"Forgetting":>12}')
print('-'*64)
print(f'{"Base (no FT)":<20} {base_acc*100:>9.1f}% {base_acc*100:>9.1f}% {"N/A":>10} {"0.0%":>12}')
for name, r in results.items():
    forget = (base_acc - r['base']) * 100
    print(f'{name:<20} {r["overall"]*100:>9.1f}% {r["base"]*100:>9.1f}% {r["new"]*100:>9.1f}% {forget:>+11.1f}%')

print(f'\nBenchmark: Fluent Speech Commands (FSC)')
print(f'This is the FIRST sub-100K param SLU model with few-shot intent addition.')
print(f'No prior work demonstrates on-device intent classification at this scale.')

# Show per-intent results for new intents
print(f'\n  New intent breakdown (NC-OPAL):')
for intent in NEW_INTENTS:
    acc = opal_per.get(intent, 0)
    n_test = len(new_test.get(intent, []))
    print(f'    {intent}: {acc*100:.1f}% ({n_test} test)')


In [ ]:
# ==============================================================================
# Cell 3: Visualization
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

methods = ['Base', 'Finetune', 'NC-OPAL']
colors = ['#888888', '#EA4335', '#34A853']

overall_accs = [base_acc*100,
                results['Finetune']['overall']*100,
                results['NC-OPAL']['overall']*100]
base_accs = [base_acc*100,
             results['Finetune']['base']*100,
             results['NC-OPAL']['base']*100]
new_accs = [0,
            results['Finetune']['new']*100,
            results['NC-OPAL']['new']*100]

# Overall
bars = axes[0].bar(methods, overall_accs, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Overall Accuracy')
axes[0].set_ylim(0, 100)
for bar, val in zip(bars, overall_accs):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.1f}', ha='center')

# Base intents
bars = axes[1].bar(methods, base_accs, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title(f'Base Intent Accuracy ({len(BASE_INTENTS)} intents)')
axes[1].set_ylim(0, 100)
for bar, val in zip(bars, base_accs):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.1f}', ha='center')

# New intents
bars = axes[2].bar(methods[1:], new_accs[1:], color=colors[1:], edgecolor='black', linewidth=0.5)
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title(f'New Intent Accuracy ({len(NEW_INTENTS)} intents, {N_SHOT}-shot)')
axes[2].set_ylim(0, 100)
for bar, val in zip(bars, new_accs[1:]):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.1f}', ha='center')

plt.suptitle('NC-SLU: Few-Shot Spoken Language Understanding (21K params)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('slu_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: slu_results.png')

# Per-intent heatmap for NC-OPAL
fig2, ax2 = plt.subplots(figsize=(16, 4))
all_intents_sorted = BASE_INTENTS + NEW_INTENTS
accs = [opal_per.get(i, 0)*100 for i in all_intents_sorted]
cs = ['#34A853' if i in NEW_INTENTS else '#4285F4' for i in all_intents_sorted]
bars = ax2.barh(range(len(all_intents_sorted)), accs, color=cs, edgecolor='black', linewidth=0.3)
ax2.set_yticks(range(len(all_intents_sorted)))
ax2.set_yticklabels([i.replace('_', ' ') for i in all_intents_sorted], fontsize=7)
ax2.set_xlabel('Accuracy (%)')
ax2.set_title('NC-OPAL: Per-Intent Accuracy (blue=base, green=new)')
ax2.set_xlim(0, 105)
ax2.invert_yaxis()
plt.tight_layout()
plt.savefig('slu_per_intent.png', dpi=150, bbox_inches='tight')
plt.show()
